In [1]:
import os
for root,dirs,files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith((".pt",".npy",".pkl")):
            print(os.path.join(root,f))

/kaggle/input/notebooks/arpitjoshua/08-final-stratified/shap_norm.npy
/kaggle/input/notebooks/arpitjoshua/08-final-stratified/mttrajnet_stratified_fold1.pt
/kaggle/input/datasets/arpitjoshua/mt-trajnet-thesis-data/kaggle_upload/assembled_trajectories.pkl


## Setup: load the saved model and the fold-1 data

SHAP attribution for all four targets, using the model saved from the authoritative run in Notebook 8. I load that model, the normalisation statistics, and the fold-1 train and test indices, so the attribution is computed on exactly the data the model was evaluated on. The same seed is fixed.

In [2]:
import numpy as np
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

SEED=42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True
torch.backends.cudnn.benchmark=False
device="cuda" if torch.cuda.is_available() else "cpu"

DATA="/kaggle/input/datasets/arpitjoshua/mt-trajnet-thesis-data/kaggle_upload"
MODEL="/kaggle/input/notebooks/arpitjoshua/08-final-stratified"

with open(DATA+"/assembled_trajectories.pkl","rb") as fh:
    data=pickle.load(fh)
X_traj=data["X_traj"]
y_arr=data["y_arr"]
groups=data["groups"]

norm=np.load(MODEL+"/shap_norm.npy",allow_pickle=True).item()
xmean=norm["xmean"];xstd=norm["xstd"];ymean=norm["ymean"];ystd=norm["ystd"]
tr_idx=norm["tr_idx"];test_idx=norm["test_idx"]

STRIDE=2
MAXLEN=6000
targets=["dissolution_av","tbl_av_hardness","tbl_rsd_weight","fct_tensile"]
sensors=["tbl_speed","fom","main_comp","tbl_fill","SREL","pre_comp","cyl_main","cyl_pre","stiffness","ejection"]

print("device:",device)
print("test batches (fold 1):",len(test_idx))
print("train batches:",len(tr_idx))
print("norm stats loaded:",xmean.shape,ymean.shape)

device: cuda
test batches (fold 1): 314
train batches: 610
norm stats loaded: (10,) (4,)


## Rebuild the model and load the saved weights

The same architecture as the authoritative run, with the weights loaded from the model saved in Notebook 8.

In [3]:
class TCNBlock(nn.Module):
    def __init__(self,in_ch,out_ch,dilation,kernel=3,dropout=0.1):
        super().__init__()
        pad=(kernel-1)*dilation
        self.conv1=nn.Conv1d(in_ch,out_ch,kernel,padding=pad,dilation=dilation)
        self.conv2=nn.Conv1d(out_ch,out_ch,kernel,padding=pad,dilation=dilation)
        self.relu=nn.ReLU()
        self.drop=nn.Dropout(dropout)
        self.pad=pad
        self.down=nn.Conv1d(in_ch,out_ch,1) if in_ch!=out_ch else None
    def forward(self,x):
        res=x if self.down is None else self.down(x)
        out=self.conv1(x)[:,:,:-self.pad]
        out=self.drop(self.relu(out))
        out=self.conv2(out)[:,:,:-self.pad]
        out=self.drop(self.relu(out))
        return self.relu(out+res)

class TCNEncoder(nn.Module):
    def __init__(self,in_ch=10,hidden=128,dropout=0.1):
        super().__init__()
        self.b1=TCNBlock(in_ch,hidden,1,dropout=dropout)
        self.b2=TCNBlock(hidden,hidden,2,dropout=dropout)
        self.b3=TCNBlock(hidden,hidden,4,dropout=dropout)
        self.b4=TCNBlock(hidden,hidden,8,dropout=dropout)
    def forward(self,x):
        x=self.b1(x);x=self.b2(x);x=self.b3(x);x=self.b4(x)
        return x.mean(dim=2)

class EvidentialHead(nn.Module):
    def __init__(self,in_dim,hidden=64,dropout=0.1):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(in_dim,hidden),nn.ReLU(),nn.Dropout(dropout),nn.Linear(hidden,4))
    def forward(self,x):
        out=self.net(x)
        gamma=out[:,0]
        nu=F.softplus(out[:,1]).clamp(min=1e-2,max=1e3)
        alpha=F.softplus(out[:,2]).clamp(min=1e-2,max=1e3)+1.0
        beta=F.softplus(out[:,3]).clamp(min=1e-2,max=1e3)
        return gamma,nu,alpha,beta

class MTTrajNet(nn.Module):
    def __init__(self,in_ch=10,hidden=128,n_targets=4,dropout=0.1):
        super().__init__()
        self.encoder=TCNEncoder(in_ch,hidden,dropout)
        self.heads=nn.ModuleList([EvidentialHead(hidden,dropout=dropout) for _ in range(n_targets)])
    def forward(self,x):
        z=self.encoder(x)
        outs=[h(z) for h in self.heads]
        gamma=torch.stack([o[0] for o in outs],dim=1)
        nu=torch.stack([o[1] for o in outs],dim=1)
        alpha=torch.stack([o[2] for o in outs],dim=1)
        beta=torch.stack([o[3] for o in outs],dim=1)
        return gamma,nu,alpha,beta

def prep_batch(traj_list,mean,std,maxlen=MAXLEN):
    out=[]
    for a in traj_list:
        a=a[::STRIDE]
        a=(a-mean)/std
        if len(a)<maxlen:
            pad=np.zeros((maxlen-len(a),a.shape[1]),dtype="float32")
            a=np.vstack([a,pad])
        else:
            a=a[:maxlen]
        out.append(a)
    arr=np.array(out,dtype="float32")
    return torch.tensor(arr).transpose(1,2)

model=MTTrajNet().to(device)
model.load_state_dict(torch.load(MODEL+"/mttrajnet_stratified_fold1.pt",map_location=device))
model.eval()

Xte=prep_batch([X_traj[i] for i in test_idx],xmean,xstd)
Xtr_bg=prep_batch([X_traj[i] for i in tr_idx[:32]],xmean,xstd)
print("model loaded, weights restored")
print("test tensor:",Xte.shape,"| background:",Xtr_bg.shape)

model loaded, weights restored
test tensor: torch.Size([314, 10, 6000]) | background: torch.Size([32, 10, 6000])


## SHAP attribution for all four targets

For each target I run GradientExplainer on the full fold-1 test set, and collect the model's epistemic uncertainty for that target. This gives, per target, an attribution map over sensors and time and an uncertainty value per batch, which I then split by confidence in the next step.

In [4]:
import shap
import time

class TargetWrapper(nn.Module):
    def __init__(self,model,target_idx):
        super().__init__()
        self.model=model
        self.t=target_idx
    def forward(self,x):
        g,_,_,_=self.model(x)
        return g[:,self.t:self.t+1]

sv_all={}
unc_all={}
t0=time.time()
for ti,tname in enumerate(targets):
    wrapper=TargetWrapper(model,ti).to(device)
    wrapper.eval()
    explainer=shap.GradientExplainer(wrapper,Xtr_bg.to(device))
    svs=[]
    for i in range(0,len(Xte),16):
        chunk=Xte[i:i+16].to(device)
        sv=explainer.shap_values(chunk)
        svs.append(np.array(sv)[:,:,:,0])
    sv_all[tname]=np.concatenate(svs,axis=0)

    unc=[]
    with torch.no_grad():
        for i in range(0,len(Xte),16):
            xb=Xte[i:i+16].to(device)
            g,n,a,b=model(xb)
            unc.append((1.0/n[:,ti]).cpu().numpy())
    unc_all[tname]=np.concatenate(unc)
    print(f"{tname}: shap {sv_all[tname].shape}, done at {round(time.time()-t0)}s")

print("\nall four targets done")

dissolution_av: shap (314, 10, 6000), done at 299s
tbl_av_hardness: shap (314, 10, 6000), done at 597s
tbl_rsd_weight: shap (314, 10, 6000), done at 895s
fct_tensile: shap (314, 10, 6000), done at 1193s

all four targets done


## Uncertainty-stratified attribution for all four targets

For each target I split the fold-1 test predictions into the most and least confident twenty percent by that target's epistemic uncertainty, then compare the attribution per sensor channel and per compression phase. The channel-level Kendall tau tests whether the model relies on different sensors when uncertain. The phase-level test has no power with three phases, so I report the phase ratios directly.

In [5]:
from scipy.stats import kendalltau

def analyze(sv,unc):
    n=len(unc);k=int(0.2*n)
    order=np.argsort(unc)
    low=order[:k];high=order[-k:]
    third=sv.shape[2]//3
    phases={"early":slice(0,third),"mid":slice(third,2*third),"late":slice(2*third,sv.shape[2])}
    def amap(idx):
        m=np.zeros((10,3))
        for pi,(pn,sl) in enumerate(phases.items()):
            m[:,pi]=np.abs(sv[idx][:,:,sl]).sum(axis=2).mean(axis=0)
        return m
    lm=amap(low);hm=amap(high)
    lc=lm.sum(axis=1);hc=hm.sum(axis=1)
    lp=lm.sum(axis=0);hp=hm.sum(axis=0)
    tau,p=kendalltau(lc,hc)
    return {"low_ch":lc,"high_ch":hc,"low_ph":lp,"high_ph":hp,"tau":tau,"p":p,
            "mean_unc_low":float(unc[low].mean()),"mean_unc_high":float(unc[high].mean())}

results={}
print(f"{'target':<18}{'tau':<8}{'p':<10}{'phase ratios (mid/late)'}")
for tname in targets:
    r=analyze(sv_all[tname],unc_all[tname])
    results[tname]=r
    mid=r["high_ph"][1]/r["low_ph"][1] if r["low_ph"][1]>0 else float("nan")
    late=r["high_ph"][2]/r["low_ph"][2] if r["low_ph"][2]>0 else float("nan")
    print(f"{tname:<18}{r['tau']:<8.3f}{r['p']:<10.4f}mid {mid:.1f}x, late {late:.1f}x")

target            tau     p         phase ratios (mid/late)
dissolution_av    0.822   0.0004    mid 1.0x, late 0.9x
tbl_av_hardness   0.822   0.0004    mid 1.7x, late 1.6x
tbl_rsd_weight    0.467   0.0726    mid 0.5x, late 1.3x
fct_tensile       0.467   0.0726    mid 1.0x, late 1.9x


In [6]:
import json
out={"analysis":"RQ3b uncertainty-stratified SHAP, all four targets, stratified cross-fitting model (Notebook 8), fold 1",
"setup":"GradientExplainer, 32 background samples, full fold-1 test set (314 batches), top/bottom 20% by per-target epistemic uncertainty; reproduced identically across two runs",
"results":{}}
for tname in targets:
    r=results[tname]
    out["results"][tname]={
    "kendall_tau":round(float(r["tau"]),3),"p":round(float(r["p"]),4),
    "significant":bool(r["p"]<0.05),
    "mean_epistemic":{"confident":round(r["mean_unc_low"],3),"uncertain":round(r["mean_unc_high"],3)},
    "channel_ratios":{sensors[i]:round(float(r["high_ch"][i]/r["low_ch"][i]),3) if r["low_ch"][i]>0 else None for i in range(10)},
    "phase_ratios":{"early":round(float(r["high_ph"][0]/r["low_ph"][0]),3),
                    "mid":round(float(r["high_ph"][1]/r["low_ph"][1]),3),
                    "late":round(float(r["high_ph"][2]/r["low_ph"][2]),3)}}
out["note"]="channel-level attribution shift is significant and reproducible on dissolution and hardness (tau 0.822, p 0.0004), not significant on weight RSD and tensile (tau 0.467, p 0.073); the strong mid and late phase-attention shift seen on the earlier model (trained on the imbalanced folds) did not replicate on the corrected stratified model, indicating that phase pattern was specific to the earlier model rather than a robust property"
with open("/kaggle/working/rq3b_all_targets_stratified.json","w") as fh:
    json.dump(out,fh,indent=2)
print(json.dumps(out,indent=2))

{
  "analysis": "RQ3b uncertainty-stratified SHAP, all four targets, stratified cross-fitting model (Notebook 8), fold 1",
  "setup": "GradientExplainer, 32 background samples, full fold-1 test set (314 batches), top/bottom 20% by per-target epistemic uncertainty; reproduced identically across two runs",
  "results": {
    "dissolution_av": {
      "kendall_tau": 0.822,
      "p": 0.0004,
      "significant": true,
      "mean_epistemic": {
        "confident": 14.642,
        "uncertain": 76.167
      },
      "channel_ratios": {
        "tbl_speed": 1.078,
        "fom": 1.028,
        "main_comp": 0.681,
        "tbl_fill": 0.798,
        "SREL": 0.581,
        "pre_comp": 1.218,
        "cyl_main": 0.71,
        "cyl_pre": 0.984,
        "stiffness": 0.581,
        "ejection": 1.678
      },
      "phase_ratios": {
        "early": 0.807,
        "mid": 0.951,
        "late": 0.896
      }
    },
    "tbl_av_hardness": {
      "kendall_tau": 0.822,
      "p": 0.0004,
      "signifi